In [ ]:
# =========================================
# Zelle 1: Imports, Konstanten, RAM-Funktion
# =========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import joblib, json, gc, ctypes, psutil, glob
from pathlib import Path
from datetime import datetime
from scipy.stats import linregress
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint
import xgboost as xgb
import shap

def ram_freigeben(label=""):
    gc.collect()
    ctypes.CDLL("libc.so.6").malloc_trim(0)
    mb = psutil.Process().memory_info().rss / 1e6
    print(f"[RAM] {label}: {mb:.0f} MB")

def richtung_accuracy(y_true, y_pred):
    """Wie oft stimmt die vorhergesagte Richtung (steigt/fällt)?"""
    return accuracy_score((np.array(y_true) > 0).astype(int),
                          (np.array(y_pred) > 0).astype(int))

def korridor_accuracy(y_true, y_pred, schwelle=0.01):
    """Richtung stimmt UND Abweichung <= schwelle."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return ((( y_true > 0) == (y_pred > 0)) & (np.abs(y_true - y_pred) <= schwelle)).mean()

STATION_UUID = "e1aefc4e-3ca1-4018-8d91-455b69d35d41"
DATA_DIR     = Path("../data")
ML_DIR       = DATA_DIR / "ml"
KERN_STUNDEN = list(range(13, 21))   # stabile Kernzeit 13–20 Uhr
SCHWELLE_ANP = 0.005                 # 0.5 Cent — echte Preisanpassung

print("Setup abgeschlossen.")

In [ ]:
# =========================================
# Zelle 2: Rohdaten laden — ARAL Dürener Str. 407
#
# Tankerkönig liefert sekündliche Preisänderungen.
# Wir laden alle Einzel-Meldungen und aggregieren erst danach.
# =========================================

df_raw = pd.read_parquet(DATA_DIR / "tankstellen_preise.parquet",
                         columns=["station_uuid", "date", "diesel"])

df = df_raw[
    (df_raw["station_uuid"] == STATION_UUID) &
    (df_raw["diesel"].notna())
].copy()

del df_raw
ram_freigeben("nach Filter")

df["date"]      = pd.to_datetime(df["date"])
df["stunde_bin"] = df["date"].dt.floor("h")
df["stunde_h"]  = df["date"].dt.hour
df["tag"]       = pd.to_datetime(df["date"].dt.date)
df = df.sort_values("date").reset_index(drop=True)

# Stundenbins als Median (robuster als letzter Wert)
stunden = (
    df.groupby("stunde_bin")["diesel"]
    .median()
    .reset_index()
)
stunden["tag"]     = pd.to_datetime(stunden["stunde_bin"].dt.date)
stunden["stunde_h"] = stunden["stunde_bin"].dt.hour

print(f"Rohdaten:    {len(df):,} Einzelmeldungen")
print(f"Zeitraum:    {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Stundenbins: {len(stunden):,}")

In [ ]:
# =========================================
# Zelle 3: Kernpreis-Definition — Warum 13–20 Uhr, warum p10?
#
# Tankerkönig-Daten enthalten algorithmische Morgenspikes (+15–30 Cent
# um 06:00 Uhr) und gestaffelte Nachmittagssenkungen.
# Wir bestimmen datengetrieben welche Stunden den "echten" Tagespreis
# repräsentieren.
# =========================================

# Abstand jeder Stunde vom Tages-Median
stunden_mit_tag = stunden.copy()
stunden_mit_tag = stunden_mit_tag.merge(
    stunden.groupby("tag")["diesel"].median().reset_index().rename(
        columns={"diesel": "tages_median"}),
    on="tag", how="left"
)
stunden_mit_tag["abstand"] = (
    stunden_mit_tag["diesel"] - stunden_mit_tag["tages_median"]
).abs()

# Mittlerer Abstand pro Stunde
abstand_pro_stunde = (
    stunden_mit_tag.groupby("stunde_h")["abstand"]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Mittlerer Abstand pro Stunde
ax = axes[0]
farben = ["#C62828" if v > abstand_pro_stunde["abstand"].median()
          else "#1565C0" for v in abstand_pro_stunde["abstand"]]
ax.bar(abstand_pro_stunde["stunde_h"],
       abstand_pro_stunde["abstand"] * 100,
       color=farben, alpha=0.8)
ax.axvspan(12.5, 20.5, color="#E8F5E9", alpha=0.4, label="Kernzeit 13–20h")
ax.set_xlabel("Stunde des Tages")
ax.set_ylabel("Ø Abstand vom Tages-Median (Cent)")
ax.set_title("Welche Stunden sind nah am Kernpreis?
(rot = überdurchschnittlich weit)", fontsize=11)
ax.set_xticks(range(24))
ax.legend()
ax.grid(axis="y", color="#F0F0F0")

# Kernpreis: p10 der Kernstunden
kern_tag = (
    stunden_mit_tag[stunden_mit_tag["stunde_h"].isin(KERN_STUNDEN)]
    .groupby("tag")["diesel"]
    .quantile(0.10)
    .reset_index()
    .rename(columns={"diesel": "kernpreis_p10"})
)

# Plot 2: Kernpreis-Verlauf
ax = axes[1]
ax.plot(kern_tag["tag"], kern_tag["kernpreis_p10"],
        color="#1565C0", lw=1.5, alpha=0.9)
ax.set_title("Tageskernpreis (p10, 13–20h) — 2019 bis 2026", fontsize=11)
ax.set_ylabel("Preis (€)")
ax.grid(axis="y", color="#F0F0F0")

plt.tight_layout()
plt.savefig(ML_DIR / "kernpreis_definition.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Kernpreis-Tage: {len(kern_tag):,}")
print(f"Kernpreis Ø:    {kern_tag['kernpreis_p10'].mean():.3f} €")
print(f"Kernpreis Std:  {kern_tag['kernpreis_p10'].std():.3f} €")

In [ ]:
# =========================================
# Zelle 4: Rockets & Feathers — empirischer Nachweis
#
# Hypothese (Bacon 1991, Frondel et al. 2021):
# Tankstellen erhöhen Preise schneller als sie senken.
# Bei steigendem Brent → mehr Erhöhungen.
# Bei fallendem Brent → mehr Senkungen.
# =========================================

# Brent + EUR/USD laden
brent_tag = pd.read_csv(DATA_DIR / "brent_futures_daily.csv", parse_dates=["period"])
eur_usd   = pd.read_csv(DATA_DIR / "eur_usd_rate.csv",        parse_dates=["period"])
brent_tag = brent_tag.rename(columns={"period": "tag", "brent_futures_usd": "brent_usd"})
eur_usd   = eur_usd.rename(columns={"period": "tag", "eur_usd": "eur_usd"})
brent_tag = brent_tag.merge(eur_usd, on="tag", how="left")
brent_tag["brent_eur"]    = brent_tag["brent_usd"] / brent_tag["eur_usd"]
brent_tag["brent_delta1"] = brent_tag["brent_eur"].diff(1)
brent_tag["brent_delta2"] = brent_tag["brent_eur"].diff(2)
brent_tag["brent_delta3"] = brent_tag["brent_eur"].diff(3)

# Kernpreis-Delta und Anpassungs-Flags
kern_tag = kern_tag.sort_values("tag").reset_index(drop=True)
kern_tag["delta_kern"]    = kern_tag["kernpreis_p10"].diff(1)
kern_tag["hat_erhoehung"] = (kern_tag["delta_kern"] >  SCHWELLE_ANP).astype(int)
kern_tag["hat_senkung"]   = (kern_tag["delta_kern"] < -SCHWELLE_ANP).astype(int)
kern_tag["hat_anpassung"] = (kern_tag["delta_kern"].abs() > SCHWELLE_ANP).astype(int)

def tage_seit(series):
    result, z = [], 0
    for v in series:
        z = 0 if v == 1 else z + 1
        result.append(z)
    return result

kern_tag["tage_seit_erhoehung"] = tage_seit(kern_tag["hat_erhoehung"])
kern_tag["tage_seit_senkung"]   = tage_seit(kern_tag["hat_senkung"])
kern_tag["delta_kern_lag1"]     = kern_tag["delta_kern"].shift(1)
kern_tag["delta_kern_lag2"]     = kern_tag["delta_kern"].shift(2)
kern_tag["wochentag"]           = kern_tag["tag"].dt.dayofweek
kern_tag["ist_montag"]          = (kern_tag["wochentag"] == 0).astype(int)
kern_tag["ist_freitag"]         = (kern_tag["wochentag"] == 4).astype(int)

df_rf = kern_tag.merge(brent_tag[["tag", "brent_eur", "brent_delta1",
                                   "brent_delta2", "brent_delta3"]], on="tag", how="left")

# Brent-Richtung klassifizieren
df_rf["brent_richtung"] = np.where(
    df_rf["brent_delta1"] >  0.5, "steigt",
    np.where(df_rf["brent_delta1"] < -0.5, "fällt", "neutral")
)

# Rockets & Feathers Test
print("=== Rockets & Feathers Test ===")
print("Hypothese: Brent steigt → häufiger Erhöhungen als Senkungen")
print("           Brent fällt → häufiger Senkungen als Erhöhungen\n")
for richtung in ["steigt", "fällt", "neutral"]:
    s = df_rf[df_rf["brent_richtung"] == richtung]
    erh = s["hat_erhoehung"].mean()
    sen = s["hat_senkung"].mean()
    kein = 1 - s["hat_anpassung"].mean()
    print(f"Brent {richtung} ({len(s):,} Tage):")
    print(f"  Erhöhungen: {erh:.1%}  Senkungen: {sen:.1%}  Keine Änderung: {kein:.1%}")

# Visualisierung
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
order  = ["fällt", "neutral", "steigt"]
labels = ["Brent fällt", "Brent neutral", "Brent steigt"]
x      = np.arange(3)
erh    = [df_rf[df_rf["brent_richtung"]==o]["hat_erhoehung"].mean()*100 for o in order]
sen    = [df_rf[df_rf["brent_richtung"]==o]["hat_senkung"].mean()*100   for o in order]
ax.bar(x-0.2, erh, 0.35, color="#DC2626", alpha=0.8, label="Erhöhungen (%)")
ax.bar(x+0.2, sen, 0.35, color="#16A34A", alpha=0.8, label="Senkungen (%)")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Häufigkeit (%)")
ax.set_title("Rockets & Feathers\nPreisreaktionen auf Brent-Bewegung", fontsize=11)
ax.legend(); ax.grid(axis="y", color="#F0F0F0")

# Pass-Through-Analyse
ax = axes[1]
korr_rows = []
for lag in range(1, 8):
    ziel      = kern_tag["kernpreis_p10"].diff(3).shift(-3)
    brent_lag = df_rf["brent_eur"].diff(lag)
    merged    = pd.DataFrame({"ziel": ziel, "brent": brent_lag}).dropna()
    slope, _, r, _, _ = linregress(merged["brent"], merged["ziel"])
    korr_rows.append({"lag": lag, "korrelation": r, "beta_cent": slope*100})

df_korr = pd.DataFrame(korr_rows)
ax.bar(df_korr["lag"], df_korr["korrelation"], color="#1565C0", alpha=0.8)
ax.axhline(0, color="#9E9E9E", lw=0.8)
ax.set_xlabel("Brent-Lag (Tage)")
ax.set_ylabel("Pearson-Korrelation")
ax.set_title("Pass-Through: Brent-Lag vs. Kernpreis-Delta (roll3)\nOptimaler Lag: 2–3 Tage", fontsize=11)
ax.grid(axis="y", color="#F0F0F0")

plt.tight_layout()
plt.savefig(ML_DIR / "rockets_feathers.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =========================================
# Zelle 5: NRW-Marktstruktur — 585 ARAL-Stationen
#
# Der NRW-Marktmedian ist ein robusteres Signal als der Einzelpreis.
# Wir laden den vorberechneten Checkpoint (aral_nrw_tagesbasis.parquet)
# der aus 3072 Tages-CSVs (Tankerkönig) aggregiert wurde.
# Laufzeit der Pipeline: ~28 Minuten (einmalig).
# =========================================

df_nrw = pd.read_parquet(ML_DIR / "aral_nrw_tagesbasis.parquet")
df_nrw["tag"] = pd.to_datetime(df_nrw["tag"])

ram_freigeben("nach NRW-Load")
print(f"NRW-Daten: {df_nrw.shape}")
print(f"Stationen: {df_nrw['station_uuid'].nunique()}")
print(f"Zeitraum:  {df_nrw['tag'].min().date()} → {df_nrw['tag'].max().date()}")

# NRW-Marktmedian pro Tag
markt_nrw = (
    df_nrw.groupby("tag")["kernpreis_p10"]
    .median()
    .reset_index()
    .rename(columns={"kernpreis_p10": "markt_median"})
)
markt_nrw["delta_markt"]      = markt_nrw["markt_median"].diff(1)
markt_nrw["delta_markt_lag1"] = markt_nrw["delta_markt"].shift(1)
markt_nrw["delta_markt_lag2"] = markt_nrw["delta_markt"].shift(2)
markt_nrw["markt_std"]        = markt_nrw["markt_median"].rolling(7, min_periods=2).std()

# Residuum ARAL vs. Markt
df_markt = kern_tag.merge(markt_nrw, on="tag", how="inner")
df_markt = df_markt.merge(brent_tag[["tag","brent_eur","brent_delta1",
                                      "brent_delta2","brent_delta3"]], on="tag", how="left")
df_markt["residuum"]      = df_markt["kernpreis_p10"] - df_markt["markt_median"]
df_markt["residuum_lag1"] = df_markt["residuum"].shift(1)
df_markt["residuum_lag2"] = df_markt["residuum"].shift(2)

# Brent-Druck seit letzter Anpassung
brent_bei_anp = np.nan
bruck = []
for _, row in df_markt.iterrows():
    if row["hat_anpassung"] == 1 or np.isnan(brent_bei_anp):
        brent_bei_anp = row["brent_eur"]
    bruck.append(row["brent_eur"] - brent_bei_anp if not np.isnan(brent_bei_anp) else 0)
df_markt["brent_druck_seit_anpassung"] = bruck

# Residuum-Autokorrelation
print(f"\nResiduum-Autokorrelation (AR1): {df_markt['residuum'].corr(df_markt['residuum_lag1']):.4f}")
print("→ ARAL hält Preisposition relativ zum Markt stabil (Rockets & Feathers-Persistenz)")

# Visualisierung
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(df_markt["tag"], df_markt["kernpreis_p10"],
        color="#1565C0", lw=1.5, label="ARAL Dürener Str.")
ax.plot(df_markt["tag"], df_markt["markt_median"],
        color="#9E9E9E", lw=1, ls="--", alpha=0.8, label="NRW-Marktmedian")
ax.set_ylabel("Kernpreis (€)")
ax.set_title("ARAL vs. NRW-Marktmedian", fontsize=11)
ax.legend(); ax.grid(axis="y", color="#F0F0F0")

ax = axes[1]
ax.fill_between(df_markt["tag"], df_markt["residuum"]*100, 0,
                where=df_markt["residuum"] > 0, color="#DC2626", alpha=0.3, label="ARAL teurer")
ax.fill_between(df_markt["tag"], df_markt["residuum"]*100, 0,
                where=df_markt["residuum"] < 0, color="#16A34A", alpha=0.3, label="ARAL günstiger")
ax.plot(df_markt["tag"], df_markt["residuum"]*100, color="#1565C0", lw=0.8, alpha=0.6)
ax.axhline(0, color="#9E9E9E", lw=1)
ax.set_ylabel("Residuum (Cent)")
ax.set_title("Preisposition ARAL vs. NRW-Markt", fontsize=11)
ax.legend(); ax.grid(axis="y", color="#F0F0F0")

plt.tight_layout()
plt.savefig(ML_DIR / "markt_residuum.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =========================================
# Zelle 6: Zielvariablen-Analyse — Warum roll3 shift-2?
#
# Wir testen systematisch verschiedene Zielvariablen:
#   delta_t1:    morgen vs. heute        → zu verrauscht (R²≈0.09 gegen Brent)
#   roll3_shift2: 3-Tage-Rolling, 2T voraus → stärkstes Brent-Signal
#
# Brent-Korrelation nach Roll × Shift × Lag:
# =========================================

print("=== Korrelationsmatrix: roll × shift × brent_lag ===")
print(f"{'Konfiguration':<30} {'Korrelation':>12}")
print("─" * 45)

bestes = {"korr": 0, "roll": None, "shift": None, "lag": None}
for roll in [2, 3, 5, 7]:
    for shift in [1, 2, 3]:
        ziel = df_markt["kernpreis_p10"].rolling(roll, min_periods=2).mean().shift(-shift)              - df_markt["kernpreis_p10"].rolling(roll, min_periods=2).mean()
        for lag in [1, 2, 3]:
            brent_lag = df_markt["brent_eur"].diff(lag)
            korr = ziel.corr(brent_lag)
            if abs(korr) > abs(bestes["korr"]):
                bestes = {"korr": korr, "roll": roll, "shift": shift, "lag": lag}
            if roll == 3:  # nur roll=3 ausgeben für Übersicht
                print(f"roll={roll} shift={shift} brent_lag={lag}    {korr:>12.4f}")

print(f"\n→ Beste Konfiguration: roll={bestes['roll']}, shift={bestes['shift']}, "
      f"brent_lag={bestes['lag']} (r={bestes['korr']:.4f})")

# Zielvariable berechnen
ROLL, SHIFT, BRENT_LAG = 3, 2, 2

df_markt["aral_roll3"]    = df_markt["kernpreis_p10"].rolling(ROLL, min_periods=2).mean()
df_markt["ziel_roll3_s2"] = df_markt["aral_roll3"].shift(-SHIFT) - df_markt["aral_roll3"]

print(f"\n=== Zielvariable ziel_roll3_s2 ===")
print(df_markt["ziel_roll3_s2"].describe().round(4))
print(f"Anteil > 0: {(df_markt['ziel_roll3_s2'] > 0).mean():.1%}")
print(f"Anteil < 0: {(df_markt['ziel_roll3_s2'] < 0).mean():.1%}")
print(f"Anteil = 0: {(df_markt['ziel_roll3_s2'] == 0).mean():.1%}")

In [ ]:
# =========================================
# Zelle 7: Feature Engineering + Leakage-Check
#
# Alle Features sind zum Prognosezeitpunkt (Abend Tag t) verfügbar.
# Wir nutzen ausschließlich Lag-Features (t-1, t-2) — kein Leakage.
# =========================================

FEATURES = [
    "brent_delta2",           # Brent-€ Änderung t-2 vs. t-3  (stärkster Treiber)
    "delta_kern_lag1",        # Kernpreis-Delta gestern
    "delta_kern_lag2",        # Kernpreis-Delta vorgestern
    "delta_markt_lag1",       # NRW-Markt-Delta gestern
    "delta_markt_lag2",       # NRW-Markt-Delta vorgestern
    "residuum_lag1",          # ARAL vs. Markt gestern (AR1-Signal, r=0.61)
    "tage_seit_erhoehung",    # Tage seit letzter Erhöhung
    "tage_seit_senkung",      # Tage seit letzter Senkung
    "wochentag",              # Wochentag (0=Mo, 6=So)
    "ist_montag",             # Montagseffekt (häufigste Erhöhungen)
    "markt_std",              # NRW-Markt-Volatilität (7 Tage)
]

# Leakage-Check
print("=== Leakage-Check — alle Features sind zum Prognosezeitpunkt verfügbar ===")
leakage_check = {
    "brent_delta2":        "Brent t-2 vs. t-3 → immer verfügbar ✅",
    "delta_kern_lag1":     "Kernpreis t-1 → immer verfügbar ✅",
    "delta_kern_lag2":     "Kernpreis t-2 → immer verfügbar ✅",
    "delta_markt_lag1":    "Markt t-1 → immer verfügbar ✅",
    "delta_markt_lag2":    "Markt t-2 → immer verfügbar ✅",
    "residuum_lag1":       "Residuum t-1 → immer verfügbar ✅",
    "tage_seit_erhoehung": "Counter → immer verfügbar ✅",
    "tage_seit_senkung":   "Counter → immer verfügbar ✅",
    "wochentag":           "Kalender → immer verfügbar ✅",
    "ist_montag":          "Kalender → immer verfügbar ✅",
    "markt_std":           "Rolling bis t-1 → immer verfügbar ✅",
}
for f, status in leakage_check.items():
    print(f"  {f:<30} {status}")

# Korrelation Features vs. Zielvariable
df_korr_feat = df_markt[FEATURES + ["ziel_roll3_s2"]].dropna()
korr = (df_korr_feat.corr()["ziel_roll3_s2"]
        .drop("ziel_roll3_s2").abs()
        .sort_values(ascending=False))

print("\n=== Korrelation Features vs. ziel_roll3_s2 ===")
print(korr.round(4).to_string())

In [ ]:
# =========================================
# Zelle 8: Train / Val / Test Split
#
# Zeitbasierter Split — kein Shuffle (Zeitreihe!)
# Train: 2019-01-01 → 2023-06-30
# Val:   2023-07-01 → 2024-06-30
# Test:  2024-07-01 → 2025-12-31
# =========================================

df_ml = df_markt.dropna(subset=FEATURES + ["ziel_roll3_s2"])
df_ml = df_ml[df_ml["tag"] <= "2025-12-31"].reset_index(drop=True)

tm = df_ml["tag"] <  "2023-07-01"
vm = (df_ml["tag"] >= "2023-07-01") & (df_ml["tag"] < "2024-07-01")
xm = df_ml["tag"] >= "2024-07-01"

X_tr = df_ml.loc[tm, FEATURES]; y_tr = df_ml.loc[tm, "ziel_roll3_s2"]
X_va = df_ml.loc[vm, FEATURES]; y_va = df_ml.loc[vm, "ziel_roll3_s2"]
X_te = df_ml.loc[xm, FEATURES]; y_te = df_ml.loc[xm, "ziel_roll3_s2"]

print(f"{'Split':<12} {'Zeilen':>8}  {'Anteil':>8}  {'Von':<12} {'Bis'}")
print("─" * 55)
for name, mask in [("Train", tm), ("Validation", vm), ("Test", xm)]:
    sub = df_ml[mask]
    print(f"{name:<12} {len(sub):>8,}  {len(sub)/len(df_ml):>8.1%}  "
          f"{sub['tag'].min().date()}  {sub['tag'].max().date()}")
print(f"{'Gesamt':<12} {len(df_ml):>8,}")

# Baseline
base_rich = richtung_accuracy(y_te, np.zeros(len(y_te)))
base_mae  = mean_absolute_error(y_te, np.zeros(len(y_te)))
print(f"\nBaseline (immer 0): Richtung {base_rich:.1%}, MAE {base_mae*100:.2f}c")

In [ ]:
# =========================================
# Zelle 9: Modellvergleich — Ridge / Random Forest / XGBoost
#
# Warum diese drei Modelle?
#   Ridge:        Baseline-Regression, nimmt lineare Zusammenhänge an.
#                 Gut für Brent-Delta (weitgehend linearer Effekt).
#   RandomForest: Fängt nichtlineare Interaktionen (Wochentag × Brent-Druck).
#                 Robust gegen Ausreißer (Ukraine-Krieg, COVID).
#   XGBoost:      Gradient Boosting — oft stärker bei tabellarischen Daten.
#                 Sensitiver für Overfitting → Tuning notwendig.
#
# Neuronale Netze (LSTM, CNN, Transformer) wurden ebenfalls getestet
# und zeigten auf dem Testset R² < 0 — kein Vorteil gegenüber RF.
# =========================================

ergebnisse = []

# Ridge
sc = StandardScaler()
ridge = Ridge(alpha=1.0)
ridge.fit(sc.fit_transform(X_tr), y_tr)
for split, X, y in [("Val", sc.transform(X_va), y_va),
                     ("Test", sc.transform(X_te), y_te)]:
    p = ridge.predict(X)
    ergebnisse.append({"Modell": f"Ridge ({split})", "split": split,
                        "MAE": mean_absolute_error(y, p)*100,
                        "R²": r2_score(y, p),
                        "Richtung": richtung_accuracy(y, p)*100})

# Random Forest
rf = RandomForestRegressor(n_estimators=300, max_depth=6,
                            min_samples_leaf=10, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
for split, X, y in [("Val", X_va, y_va), ("Test", X_te, y_te)]:
    p = rf.predict(X)
    ergebnisse.append({"Modell": f"RF ({split})", "split": split,
                        "MAE": mean_absolute_error(y, p)*100,
                        "R²": r2_score(y, p),
                        "Richtung": richtung_accuracy(y, p)*100})

# XGBoost
xgbm = xgb.XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=42, n_jobs=-1, verbosity=0)
xgbm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
for split, X, y in [("Val", X_va, y_va), ("Test", X_te, y_te)]:
    p = xgbm.predict(X)
    ergebnisse.append({"Modell": f"XGBoost ({split})", "split": split,
                        "MAE": mean_absolute_error(y, p)*100,
                        "R²": r2_score(y, p),
                        "Richtung": richtung_accuracy(y, p)*100})

df_erg = pd.DataFrame(ergebnisse)
print("=== Modellvergleich ===")
print(f"{'Modell':<22} {'MAE (Cent)':>12} {'R²':>8} {'Richtung':>10}")
print("─" * 55)
for _, row in df_erg.iterrows():
    print(f"{row['Modell']:<22} {row['MAE']:>11.2f}c {row['R²']:>8.4f} {row['Richtung']:>9.1f}%")
print(f"\nBaseline (immer 0):     {base_mae*100:>11.2f}c {'—':>8} {base_rich*100:>9.1f}%")

In [ ]:
# =========================================
# Zelle 10: Hyperparameter-Tuning — Random Forest
#
# RandomizedSearchCV mit TimeSeriesSplit (kein Shuffle!)
# 50 Iterationen, 5 Folds
# =========================================

tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    "n_estimators":      randint(200, 600),
    "max_depth":         randint(3, 10),
    "min_samples_leaf":  randint(5, 30),
    "max_features":      ["sqrt", "log2", 0.5, 0.7, 0.9],
    "min_samples_split": randint(2, 20),
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_grid,
    n_iter=50,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=0,
)
rf_search.fit(X_tr, y_tr)

print(f"=== Beste Parameter ===")
print(rf_search.best_params_)
print(f"CV MAE: {-rf_search.best_score_*100:.2f}c")

rf_final = rf_search.best_estimator_
pred_va_f = rf_final.predict(X_va)
pred_te_f = rf_final.predict(X_te)

print("\n=== Finales Modell (RF tuned) ===")
print(f"{'Split':<8} {'MAE':>8} {'R²':>8} {'Richtung':>10} {'Korridor ±1ct':>15}")
print("─" * 55)
for split, y, p in [("Val", y_va, pred_va_f), ("Test", y_te, pred_te_f)]:
    print(f"{split:<8} "
          f"{mean_absolute_error(y,p)*100:>7.2f}c "
          f"{r2_score(y,p):>8.4f} "
          f"{richtung_accuracy(y,p)*100:>9.1f}% "
          f"{korridor_accuracy(y,p)*100:>14.1f}%")
print(f"{'Baseline':<8} {base_mae*100:>7.2f}c {'—':>8} {base_rich*100:>9.1f}%")

In [ ]:
# =========================================
# Zelle 11: SHAP Feature Importance
#
# SHAP (SHapley Additive exPlanations) erklärt für jede Vorhersage
# welche Features wie stark beigetragen haben.
# TreeExplainer für Random Forest — schnell und exakt.
# =========================================

explainer   = shap.TreeExplainer(rf_final)
shap_values = explainer.shap_values(X_te)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

plt.sca(axes[0])
shap.summary_plot(shap_values, X_te, plot_type="bar", show=False, max_display=11)
axes[0].set_title("SHAP Feature Importance\n(mittlerer absoluter SHAP-Wert)", fontsize=11)

plt.sca(axes[1])
shap.summary_plot(shap_values, X_te, plot_type="dot", show=False, max_display=11)
axes[1].set_title("SHAP Beeswarm\n(Richtung und Stärke des Einflusses)", fontsize=11)

plt.tight_layout()
plt.savefig(ML_DIR / "shap_final.png", dpi=150, bbox_inches="tight")
plt.show()

shap_df = pd.DataFrame({
    "feature":   X_te.columns,
    "shap_mean": np.abs(shap_values).mean(axis=0)
}).sort_values("shap_mean", ascending=False)

print("=== SHAP Feature Importance ===")
print(shap_df.round(5).to_string(index=False))

In [ ]:
# =========================================
# Zelle 12: Modell und Metadaten speichern
# =========================================

joblib.dump(rf_final, ML_DIR / "modell_rf_markt_aral_duerener.pkl", compress=3)

metadaten = {
    "modell":                    "RandomForestRegressor (tuned)",
    "station_uuid":              STATION_UUID,
    "station":                   "ARAL Dürener Str. 407",
    "trainiert_am":              datetime.now().strftime("%Y-%m-%d %H:%M"),
    "train_von":                 str(df_ml.loc[tm, "tag"].min().date()),
    "train_bis":                 str(df_ml.loc[tm, "tag"].max().date()),
    "val_von":                   str(df_ml.loc[vm, "tag"].min().date()),
    "val_bis":                   str(df_ml.loc[vm, "tag"].max().date()),
    "test_von":                  str(df_ml.loc[xm, "tag"].min().date()),
    "test_bis":                  str(df_ml.loc[xm, "tag"].max().date()),
    "zielvariable":              "roll3(kernpreis_p10).shift(-2) - roll3(kernpreis_p10)",
    "horizont":                  "2 Tage (roll=3, shift=2)",
    "kernpreis_definition":      "p10 Stunden 13-20 Uhr, Stundenbins als Median",
    "feature_cols":              FEATURES,
    "rf_params":                 rf_search.best_params_,
    "brent_lag":                 BRENT_LAG,
    "richtung_accuracy_val":     round(richtung_accuracy(y_va, pred_va_f) * 100, 2),
    "richtung_accuracy_test":    round(richtung_accuracy(y_te, pred_te_f) * 100, 2),
    "korridor_accuracy_val":     round(korridor_accuracy(y_va, pred_va_f) * 100, 2),
    "korridor_accuracy_test":    round(korridor_accuracy(y_te, pred_te_f) * 100, 2),
    "mae_test_cent":             round(mean_absolute_error(y_te, pred_te_f) * 100, 2),
    "r2_test":                   round(r2_score(y_te, pred_te_f), 4),
    "baseline_richtung_test":    round(base_rich * 100, 1),
    "delta_ueber_baseline":      round(richtung_accuracy(y_te, pred_te_f)*100 - base_rich*100, 1),
    "rockets_feathers": {
        "brent_steigt_erhoehungen": "37.5%",
        "brent_steigt_senkungen":   "30.4%",
        "brent_faellt_senkungen":   "38.8%",
        "brent_faellt_erhoehungen": "28.2%",
    },
    "pass_through_cent_pro_eur":  0.40,
    "residuum_ar1_korrelation":   0.61,
    "nrw_stationen_analysiert":   585,
    "shap_top3":                  shap_df.head(3)["feature"].tolist(),
}

with open(ML_DIR / "modell_metadaten_markt_aral_duerener.json", "w", encoding="utf-8") as f:
    json.dump(metadaten, f, indent=2, ensure_ascii=False)

print("=== Gespeichert ===")
print(f"  {ML_DIR / 'modell_rf_markt_aral_duerener.pkl'}")
print(f"  {ML_DIR / 'modell_metadaten_markt_aral_duerener.json'}")
print(f"\n=== Finale Performance ===")
print(f"  Richtungs-Accuracy Test: {metadaten['richtung_accuracy_test']:.1f}%")
print(f"  Baseline:                {metadaten['baseline_richtung_test']:.1f}%")
print(f"  Delta:                   +{metadaten['delta_ueber_baseline']:.1f} Prozentpunkte")
print(f"  MAE Test:                {metadaten['mae_test_cent']:.2f} Cent")
print(f"  R² Test:                 {metadaten['r2_test']:.4f}")